In [ ]:
"This IPYNB file is only used as a test, and a tool for deploying the model. The real Code goes in the same file, but "
"with a .py extension rather than a .ipynb"

In [1]:
from pathlib import Path
import pandas as pd

# Try common working directories (notebook folder and workspace root).
candidate_paths = [
    Path("../../../../SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
    Path("backend/SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate usgs_data_USGS-01646500.csv")
print(f"Using CSV file at: {csv_path}")

df = pd.read_csv(csv_path)
df

Using CSV file at: ..\..\..\..\SOURCES_AND_DATASHEETS\usgs_data_USGS-01646500.csv


,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,snowfall,snow_depth,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,temperature_2m,wind_speed_10m,vapour_pressure_deficit,precip_3hr,precip_24hr,precip_72hr
0,2010-07-06 00:00:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-07-06 00:15:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-07-06 00:30:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-07-06 00:45:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-07-06 01:00:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626707,2026-07-06 23:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,1.0,1.0,0.0,0.0,NaN,NaN,24.65,0.648999,0.118415,29.8,50.2,74.800001
626708,2026-07-07 00:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,1.8,1.8,0.0,0.0,NaN,NaN,24.10,11.074022,0.062441,31.6,51.4,76.600000
626709,2026-07-07 01:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,0.7,0.7,0.0,0.0,NaN,NaN,23.45,11.225132,0.051792,32.3,51.5,77.300000
626710,2026-07-07 02:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,1.1,1.1,0.0,0.0,NaN,NaN,22.85,10.137692,0.016814,33.4,52.0,78.400000


In [2]:
# Flood Action Stage: 5 ft
# Minor Flood Stage: 10 ft
# Moderate Flood Stage: 12 ft
# Major Flood Stage: 14 ft
FLOOD_ACTION_STAGE = 5.0
MINOR_FLOOD_STAGE = 10.0
MODERATE_FLOOD_STAGE = 12.0
MAJOR_FLOOD_STAGE = 14.0

#df.hist(figsize=(10, 6))
# get the instances where Gage Height is > 5
df = df.dropna(subset=['gage_height_ft'])
df_flood = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE]
df_minor_flood = df[df['gage_height_ft'] > MINOR_FLOOD_STAGE]
df_moderate_flood = df[df['gage_height_ft'] > MODERATE_FLOOD_STAGE]
df_major_flood = df[df['gage_height_ft'] > MAJOR_FLOOD_STAGE]

# print the lengths of each of the dataframes
print(f"Total records: {len(df)}")
print(f"Flood Action Stage records: {len(df_flood)}")
print(f"Minor flood records: {len(df_minor_flood)}")
print(f"Moderate flood records: {len(df_moderate_flood)}")
print(f"Major flood records: {len(df_major_flood)}")

Total records: 624474
Flood Action Stage records: 66631
Minor flood records: 1199
Moderate flood records: 55
Major flood records: 0


In [3]:

# Name the 'Unnamed: 0' column as 'datetime' and convert it to datetime type
df['datetime'] = pd.to_datetime(df['Unnamed: 0'])

df = df.sort_values('datetime').reset_index(drop=True)

# how many hours of gap counts as "the storm ended" (tune this to your data's
# sampling frequency, e.g. 6-12h for hourly gauge data, 24-48h for daily)
GAP_HOURS = 12

# isolate just the flagged (action-stage) rows
flood_rows = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE].copy()

# time since previous flagged reading, saved in column 'gap'
flood_rows['gap'] = flood_rows['datetime'].diff()

# start a new event whenever the gap exceeds threshold (or it's the first row)
flood_rows['new_event'] = (
    flood_rows['gap'].isna() | (flood_rows['gap'] > pd.Timedelta(hours=GAP_HOURS))
)
flood_rows['event_id'] = flood_rows['new_event'].cumsum()

df = df.merge(
    flood_rows[['datetime', 'event_id']],
    on='datetime',
    how='left'
)

# summarize each event
events = flood_rows.groupby('event_id').agg(
    start=('datetime', 'min'),
    end=('datetime', 'max'),
    n_readings=('datetime', 'count'),
    peak_gage_height=('gage_height_ft', 'max')  # adjust column name as needed
).reset_index(drop=True)

events['duration_hours'] = (events['end'] - events['start']).dt.total_seconds() / 3600

print(f"Total flagged readings: {len(flood_rows)}")
print(f"Independent storm events: {len(events)}")
print(events)


C:\Users\drpri\AppData\Local\Temp\ipykernel_18408\2735280843.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['datetime'] = pd.to_datetime(df['Unnamed: 0'])


Total flagged readings: 66631
Independent storm events: 123
                        start                       end  n_readings  \
0   2010-12-03 00:00:00+00:00 2010-12-05 12:15:00+00:00         242   
1   2011-02-27 11:30:00+00:00 2011-03-04 14:45:00+00:00         494   
2   2011-03-07 01:30:00+00:00 2011-03-19 02:30:00+00:00        1153   
3   2011-03-26 06:30:00+00:00 2011-03-28 07:30:00+00:00         194   
4   2011-04-13 08:45:00+00:00 2011-05-08 04:15:00+00:00        2382   
..                        ...                       ...         ...   
118 2025-06-10 16:00:00+00:00 2025-06-13 06:00:00+00:00         193   
119 2025-06-16 22:15:00+00:00 2025-06-22 17:45:00+00:00         544   
120 2025-07-19 23:50:00+00:00 2025-07-20 01:35:00+00:00          22   
121 2026-02-21 15:35:00+00:00 2026-02-24 23:50:00+00:00         964   
122 2026-05-24 18:55:00+00:00 2026-05-31 21:10:00+00:00        2044   

     peak_gage_height  duration_hours  
0                6.70           60.25  
1      

In [4]:
LOOKAHEAD_HOURS = 24  # predict within next 24h
FREQ_MINUTES = 15     # Hydraulic data's sampling interval 

# Cqalculate how many rows ahead
lookahead_steps = int(LOOKAHEAD_HOURS * 60 / FREQ_MINUTES)

# 'will_flood' checks whether the flood stage will be exceeded in the next `lookahead_steps` readings, setting it as 
# 1 if any of the next readings exceed the flood action stage, and 0 otherwise. This is used as the target variable for the model.

df['will_flood'] = (
    df['gage_height_ft']
    .shift(-1)                                   
    .rolling(window=lookahead_steps, min_periods=1)
    .max()
    .shift(-(lookahead_steps - 1))                # align window to start right after current row
    > FLOOD_ACTION_STAGE
).astype(int)



In [5]:
import numpy as np

# make a column called gage_height_roc_1h and gage_height_roc_6h to see the rate of change in gage height over 1 hour and 6 hours, respectively 
df['gage_height_roc_1h'] = df['gage_height_ft'].diff(int(60/FREQ_MINUTES))
df['gage_height_roc_6h'] = df['gage_height_ft'].diff(int(360/FREQ_MINUTES))

df['gage_height_now'] = df['gage_height_ft']
df['streamflow_now'] = df['streamflow_cfs']

for col in ['precip_3hr', 'precip_24hr', 'precip_72hr', 'turbidity_fnu']:
    if col in df.columns:
        df[f'{col}_log'] = np.log1p(df[col].clip(lower=0))

feature_columns = [
    # hydraulic — current state + trend
    'gage_height_ft',
    'gage_height_roc_1h',
    'gage_height_roc_6h',

    # precip — log-transformed versions only (not the raw skewed ones)
    'precip_3hr_log',
    'precip_24hr_log',
    'precip_72hr_log',

    # weather
    'temperature_2m',
    'wind_speed_10m',
    'vapour_pressure_deficit',
    'rain',
    'snowfall',
    'snow_depth',

    # water quality
    'specific_conductance_us_cm',
    'temperature_c',
]


In [6]:
# tag every row (not just flood rows) with which event's "influence window" it falls in
# so the same storm doesn't appear in both train and test
events_sorted = events.sort_values('start').reset_index(drop=True)

# hold out the most recent ~20% of events as test
n_test_events = int(len(events_sorted) * 0.2)
test_events = events_sorted.iloc[-n_test_events:]
train_events = events_sorted.iloc[:-n_test_events]

test_start_cutoff = test_events['start'].min() - pd.Timedelta(days=3)  # buffer before first test storm

train_df = df[df['datetime'] < test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])
test_df  = df[df['datetime'] >= test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])

test_df

,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,...,event_id,will_flood,gage_height_roc_1h,gage_height_roc_6h,gage_height_now,streamflow_now,precip_3hr_log,precip_24hr_log,precip_72hr_log,turbidity_fnu_log
430325,2022-12-14 13:30:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,-0.01,-0.03,3.13,3520.0,0.000000,0.832909,3.234749,0.875469
430326,2022-12-14 13:45:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.788457,3.234749,0.875469
430327,2022-12-14 14:00:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.741937,3.234749,0.875469
430328,2022-12-14 14:15:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.693147,3.234749,0.875469
430329,2022-12-14 14:30:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.02,3.13,3520.0,0.000000,0.641854,3.234749,0.875469
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
624469,2026-07-05 23:40:00+00:00,2.82,1980.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,-0.01,-0.01,2.82,1980.0,1.757858,3.063391,3.828641,1.945910
624470,2026-07-05 23:45:00+00:00,2.82,1980.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,-0.01,-0.01,2.82,1980.0,1.757858,3.063391,3.828641,1.945910
624471,2026-07-05 23:50:00+00:00,2.82,1980.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,0.00,-0.01,2.82,1980.0,1.757858,3.063391,3.828641,1.945910
624472,2026-07-05 23:55:00+00:00,2.82,1980.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,0.00,-0.01,2.82,1980.0,1.757858,3.063391,3.828641,1.945910
